In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from flonacomldft.utils.io_utils import get_path, get_project_path
import json
import os
import pandas as pd

In [3]:
# paths
DFT_DATA_PATH = get_path() + '/andersen/'
DFT_SIMS_PATH = get_project_path()


In [4]:
# simulations args files
idxs = [3054029, 3074955, 3054425, 3075074]

args_paths = {}

for idx in idxs[:2]:
    args_paths[idx] = [
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'0/args_multimodal_sampling_'+str(idx)+'0.json',
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'1/args_multimodal_sampling_'+str(idx)+'1.json',
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'2/args_multimodal_sampling_'+str(idx)+'2.json',
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'3/args_multimodal_sampling_'+str(idx)+'3.json',
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'4/args_multimodal_sampling_'+str(idx)+'4.json',
              '4-multimodal/results_multimodal_sampling_mixture_'+str(idx)+'5/args_multimodal_sampling_'+str(idx)+'5.json'
              ]

for idx in idxs[2:]:
    args_paths[idx] = [
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'0/args_multimodal_sampling_'+str(idx)+'0.json',
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'1/args_multimodal_sampling_'+str(idx)+'1.json',
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'2/args_multimodal_sampling_'+str(idx)+'2.json',
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'3/args_multimodal_sampling_'+str(idx)+'3.json',
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'4/args_multimodal_sampling_'+str(idx)+'4.json',
              '5-multimodal-mlp/results_multimodal_sampling_mixture_'+str(idx)+'5/args_multimodal_sampling_'+str(idx)+'5.json'
              ]

# Loading args files

In [5]:
args_df = []

for idx in idxs:
    for i in range(args_paths[idx].__len__()):
        path = DFT_SIMS_PATH + '/' + args_paths[idx][i]
        if not os.path.exists(path):
            continue
        args = json.load(open(DFT_SIMS_PATH + '/' + args_paths[idx][i], 'r'))
        for key in args.keys():
            if isinstance(args[key], list):
                args[key] = str(args[key])

        df = pd.DataFrame(args, index=[i])
        df['disbatch_id'] = str(df['process_id'].values)[1:-2]

        args_df.append(df)

all_args_df = pd.concat(args_df)

# Energy type comparison

In [6]:
all_args_df[['random_seed', 'process_id', 'energy_type', 'disbatch_id']]

,random_seed,process_id,energy_type,disbatch_id
2,27776,30540292,dft,3054029
4,86002,30540294,dft,3054029
5,13007,30540295,dft,3054029
5,74691,30749555,dft,3074955
0,13794,30544250,dft-mlp,3054425
1,34151,30544251,dft-mlp,3054425
2,96315,30544252,dft-mlp,3054425
3,86577,30544253,dft-mlp,3054425
4,96577,30544254,dft-mlp,3054425
5,46622,30544255,dft-mlp,3054425


# Deleting columns with unique values

In [7]:
dfs = []
for i in range(6):
    df = all_args_df[all_args_df.index==i]
    columns_to_drop = [col for col in df.columns if df[col].nunique() == 1]
    for col in ['energy_type', 'flow_ids']:
        if col in columns_to_drop:
            columns_to_drop.remove(col)
    df.drop(columns_to_drop, axis=1, inplace=True)
    dfs.append(df)

dfs = pd.concat(dfs)

# Energy type DFT

In [8]:
dfs[dfs['energy_type'] == 'dft']

,threads,process_id,random_seed,energy_type,mlps_id,date_start,date_end,disbatch_id,flow_type,flows_id,train_mlp_models
2,None,30540292,27776,dft,None,20231209135948,20231215132942,3054029,1-adaptive,"[29941690, 29941694]",NaN
4,None,30540294,86002,dft,None,20231209135941,20231216085403,3054029,1-adaptive,"[29941693, 29941697]",NaN
5,None,30540295,13007,dft,None,20231209135941,20231214064527,3054029,1-adaptive,"[29941693, 29941697]",NaN
5,None,30749555,74691,dft,None,20231218182626,20231223115003,3074955,1-adaptive,"[29941693, 29941697]",False


In [9]:
dfs[dfs['energy_type'] == 'dft-mlp']

,threads,process_id,random_seed,energy_type,mlps_id,date_start,date_end,disbatch_id,flow_type,flows_id,train_mlp_models
0,None,30544250,13794,dft-mlp,None,20231209223144,20231213032132,3054425,NaN,NaN,NaN
0,None,30750740,46807,dft-mlp,None,20231218183308,20231221031537,3075074,NaN,NaN,NaN
1,None,30544251,34151,dft-mlp,None,20231209223144,20231211130159,3054425,NaN,NaN,NaN
1,None,30750741,58764,dft-mlp,None,20231218183308,20231220075955,3075074,NaN,NaN,NaN
2,None,30544252,96315,dft-mlp,None,20231209223144,20231211185809,3054425,2-adaptive-mlp,"[30019620, 30019624]",NaN
2,None,30750742,85238,dft-mlp,None,20231218183308,20231220093753,3075074,2-adaptive-mlp,"[30019620, 30019624]",NaN
3,None,30544253,86577,dft-mlp,None,20231209223144,20231211114421,3054425,NaN,NaN,NaN
3,None,30750743,34779,dft-mlp,None,20231218183308,20231220194232,3075074,NaN,NaN,NaN
4,None,30544254,96577,dft-mlp,None,20231209223143,20231211075651,3054425,2-adaptive-mlp,"[30019623, 30019627]",NaN
4,None,30750744,35735,dft-mlp,None,20231218183306,20231220034421,3075074,2-adaptive-mlp,"[30019623, 30019627]",NaN


# Params for simulations

In [10]:
dfs[dfs['disbatch_id'] == str(idxs[0])]

,threads,process_id,random_seed,energy_type,mlps_id,date_start,date_end,disbatch_id,flow_type,flows_id,train_mlp_models
2,None,30540292,27776,dft,None,20231209135948,20231215132942,3054029,1-adaptive,"[29941690, 29941694]",NaN
4,None,30540294,86002,dft,None,20231209135941,20231216085403,3054029,1-adaptive,"[29941693, 29941697]",NaN
5,None,30540295,13007,dft,None,20231209135941,20231214064527,3054029,1-adaptive,"[29941693, 29941697]",NaN


## missing random_seed

In [15]:
rand_seeds = []

for i in range(6):
    path = get_project_path() + '/' + '/'.join(
        args_paths[idxs[0]][i].split('/')[:1]
        ) + '/logs/' + str(idxs[0]) + '_' +str(i)+'.log'

    rand_seed = !cat $path | grep 'seed'
    rand_seed = int(rand_seed[0].split(' ')[-1][:-1])
    rand_seeds.append(rand_seed)

rand_seeds

[7800, 26159, 27776, 567, 86002, 13007]

In [12]:
dfs[dfs['disbatch_id'] == str(idxs[0])]['random_seed']

2    27776
4    86002
5    13007
Name: random_seed, dtype: int64

In [13]:
dfs[dfs['disbatch_id'] == str(idxs[2])]

,threads,process_id,random_seed,energy_type,mlps_id,date_start,date_end,disbatch_id,flow_type,flows_id,train_mlp_models
0,None,30544250,13794,dft-mlp,None,20231209223144,20231213032132,3054425,NaN,NaN,NaN
1,None,30544251,34151,dft-mlp,None,20231209223144,20231211130159,3054425,NaN,NaN,NaN
2,None,30544252,96315,dft-mlp,None,20231209223144,20231211185809,3054425,2-adaptive-mlp,"[30019620, 30019624]",NaN
3,None,30544253,86577,dft-mlp,None,20231209223144,20231211114421,3054425,NaN,NaN,NaN
4,None,30544254,96577,dft-mlp,None,20231209223143,20231211075651,3054425,2-adaptive-mlp,"[30019623, 30019627]",NaN
5,None,30544255,46622,dft-mlp,None,20231209223143,20231211074532,3054425,2-adaptive-mlp,"[30019623, 30019627]",True


In [14]:
dfs[dfs['disbatch_id'] == str(idxs[2])]['random_seed']

0    13794
1    34151
2    96315
3    86577
4    96577
5    46622
Name: random_seed, dtype: int64

In [16]:
rand_seeds = []

for i in range(6):
    path = get_project_path() + '/' + '/'.join(
        args_paths[idxs[2]][i].split('/')[:1]
        ) + '/logs/' + str(idxs[2]) + '_' +str(i)+'.log'

    rand_seed = !cat $path | grep 'seed'
    rand_seed = int(rand_seed[0].split(' ')[-1][:-1])
    rand_seeds.append(rand_seed)

rand_seeds

[13794, 34151, 96315, 86577, 96577, 46622]